# CottonLens AI · Colab
Select **T4 GPU**, then **Run all**. This one notebook mounts Drive, creates isolated Python 3.12, runs compatibility tests, audits data, trains four 126-session walk-forward folds (Naive, Ridge, XGBoost, LSTM), shows LSTM epochs, performs a previously observed historical audit, and validates the ZIP. CFTC is excluded from model inputs until publication timestamps are verified. All heavy training stays in Colab. A full run can take hours; completed checkpoints remain on Drive.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/CottonLensAI')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
import subprocess
import sys
REPO = Path('/content/CottonLensAI')
URL = 'https://github.com/ErayKulkizaga/CottonLensAI.git'
def git(*args):
    result = subprocess.run(['git', *map(str, args)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout, flush=True)
    if result.returncode:
        raise RuntimeError('Git failed; see full output above.')
    return result.stdout.strip()
if (REPO / '.git').exists():
    assert git('-C', REPO, 'remote', 'get-url', 'origin') == URL, 'Unexpected repository remote'
    assert not git('-C', REPO, 'status', '--porcelain'), 'Save your Colab repo edits before updating'
    git('-C', REPO, 'pull', '--ff-only')
else:
    git('clone', URL, REPO)
sys.path.insert(0, str(REPO / 'ml'))
import importlib, colab_setup
importlib.reload(colab_setup)
from colab_setup import setup, run, INFERENCE
git('-C', REPO, 'rev-parse', 'HEAD')

In [ ]:
PYTHON = setup()  # uv sync --locked; Python 3.12.11; dedicated /content venvs
print('Training interpreter:', PYTHON)
print('Inference-only interpreter:', INFERENCE / 'bin/python')

In [ ]:
SMOKE_PASSED = False
SMOKE_REPORT = DRIVE_ROOT / 'reports/environment-smoke.json'
run([PYTHON, '-m', 'cottonlens_ml.smoke', '--repo', REPO,
     '--report', SMOKE_REPORT, '--inference-python', INFERENCE / 'bin/python'])
SMOKE_PASSED = True

In [ ]:
DATA_PASSED = False
assert SMOKE_PASSED, 'Run the smoke cell successfully first'
# Reuse a successful Drive cache. Pass --refresh only for an intentional final data refresh.
run([PYTHON, '-m', 'cottonlens_ml.prepare', '--drive-root', DRIVE_ROOT])
import json
quality = json.loads((DRIVE_ROOT / 'data/processed/data_quality.json').read_text())
print('Source first dates:', quality['source_first_dates'])
print('Modeling first date:', quality['modeling_first_date'])
print('Invalid values:', quality['invalid_market_value_count'])
print('Incomplete current-day rows excluded:', quality['incomplete_current_day_rows_excluded'])
print('CFTC policy:', quality['cftc_model_policy'])
DATA_PASSED = True

In [ ]:
import json
TRAIN_PASSED = False
assert SMOKE_PASSED and DATA_PASSED, 'Run smoke and source validation first'
assert json.loads(SMOKE_REPORT.read_text())['status'] == 'passed', 'Smoke tests must pass first'
# Uses validated Drive cache. Fold progress, trial settings and LSTM epochs stream below.
run([PYTHON, '-m', 'cottonlens_ml.pipeline', '--drive-root', DRIVE_ROOT])
TRAIN_PASSED = True

In [ ]:
assert TRAIN_PASSED, 'Training/export must finish successfully first'
run([PYTHON, '-m', 'cottonlens_ml.validate_release', '--repo', REPO,
     '--drive-root', DRIVE_ROOT, '--inference-python', INFERENCE / 'bin/python'])
release = (DRIVE_ROOT / 'artifacts/releases/latest.txt').read_text().strip()
print('Validated ZIP:', DRIVE_ROOT / 'artifacts/releases' / release)
print('Checksum:', DRIVE_ROOT / 'artifacts/releases' / (release + '.sha256'))
import zipfile
with zipfile.ZipFile(DRIVE_ROOT / 'artifacts/releases' / release) as archive:
    metrics_name = next(name for name in archive.namelist() if name.endswith('/metrics.json'))
    rows = json.loads(archive.read(metrics_name))
    for row in rows:
        evidence = row.get('walkforward', {})
        print(row['model'], 'T+' + str(row['horizon']), 'walk-forward MAE:', round(evidence.get('mae', 0), 3), 'n:', evidence.get('sample_count'), 'selected:', row['selected'])